# Afterburner Submission Notebook

**Stage:** Afterburner  
**Plan:** `docs/afterburner.md`  
**Primary output:** `submission_afterburner.csv`

Extends Shockwave with destination/star price normalization, search context and dispersion features, destination/star and hotel/window priors, auxiliary click models, longer CatBoost, signed blending, and an OOF ranker stacker.


In [1]:
from pathlib import Path
import sys
import gc
import json
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils import (
    compute_ndcg_fast,
    load_test,
    load_train,
    make_submission,
    train_val_split,
)
from src.final_pipeline import (
    build_test_feature_frame,
    build_validation_feature_frames,
    get_feature_cols,
    make_relevance,
    minmax_normalize_per_search,
    optimize_blend_weights,
    optimize_signed_blend_weights,
    rrf_score,
    save_run_json,
    sort_for_group_model,
    z_normalize_per_search,
)

try:
    from catboost import CatBoostRanker, Pool
except Exception as exc:
    CatBoostRanker = None
    Pool = None
    print(f"CatBoost unavailable: {exc}")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
pd.set_option("display.max_columns", 200)
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


## Configuration

For a quick smoke test, set `USE_SAMPLE=True`. For the real validation and final submission, keep `USE_SAMPLE=False`. Leave `RUN_FINAL_SUBMISSION=False` until validation selects the winning ensemble.


In [2]:
RANDOM_STATE = 42
USE_SAMPLE = False
SAMPLE_FRAC = 0.15
SVD_COMPONENTS = 20

TRY_LGB_GPU = True
LGB_AUTO_FALLBACK_TO_CPU = True
RUN_RANK_XENDCG = True
RUN_UNBIASED_XENDCG = True
RUN_CATBOOST = True
RUN_META_STACKER = True

CATBOOST_TASK_TYPE = "GPU"  # switch to "CPU" if CUDA is unavailable
CATBOOST_ITERATIONS = 6000
CATBOOST_EARLY_STOPPING = 300
CATBOOST_KEEP_SVD_FACTORS = False
CATBOOST_USE_ID_CATEGORICALS = True

META_N_SPLITS = 5
META_NUM_BOOST_ROUND = 2000
META_EARLY_STOPPING = 75

RUN_FINAL_SUBMISSION = False
USE_TRAIN_TEST_COUNTS = False  # True is legal/transductive for counts only, but changes final feature distribution

PW_NUM_BOOST_ROUND = 9000
PW_EARLY_STOPPING = 150
CLICK_NUM_BOOST_ROUND = 5000
CLICK_EARLY_STOPPING = 120
CLICK_ONLY_NUM_BOOST_ROUND = 5000
CLICK_ONLY_EARLY_STOPPING = 120
LR_NUM_BOOST_ROUND = 10000
LR_EARLY_STOPPING = 250
XENDCG_NUM_BOOST_ROUND = 7000
XENDCG_EARLY_STOPPING = 200
UNBIASED_XENDCG_NUM_BOOST_ROUND = 5000
UNBIASED_XENDCG_EARLY_STOPPING = 200


In [3]:
def load_run_json(name):
    path = OUTPUT_DIR / name
    if not path.exists():
        return {}
    with path.open() as f:
        return json.load(f)


def sanitize_lgb_params(params, objective, metric=None):
    clean = dict(params)
    for key in ["best_iteration", "best_score", "val_ndcg", "val_logloss", "notes", "feature_importance"]:
        clean.pop(key, None)
    clean["objective"] = objective
    if metric is not None:
        clean["metric"] = metric
    clean.setdefault("feature_pre_filter", False)
    clean.setdefault("verbose", -1)
    clean.setdefault("seed", RANDOM_STATE)
    clean.setdefault("num_threads", -1)
    return clean


gpu_params = {"device": "gpu", "gpu_use_dp": False} if TRY_LGB_GPU else {}
print("LightGBM device params:", gpu_params)

pw_run = load_run_json("pointwise_run2.json")
lr_run = load_run_json("lambdarank_run2.json")

PW_PARAMS = sanitize_lgb_params(
    pw_run.get("best_params", {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="binary",
    metric="binary_logloss",
)
PW_PARAMS.update(gpu_params)

CLICK_PARAMS = sanitize_lgb_params(
    {
        "objective": "binary",
        "metric": "binary_logloss",
        "num_leaves": 255,
        "learning_rate": 0.03,
        "min_child_samples": 150,
        "lambda_l1": 0.05,
        "lambda_l2": 1.0,
        "feature_fraction": 0.7,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    },
    objective="binary",
    metric="binary_logloss",
)
CLICK_PARAMS.update(gpu_params)

CLICK_ONLY_PARAMS = dict(CLICK_PARAMS)
CLICK_ONLY_PARAMS.update({"learning_rate": 0.025, "num_leaves": 255})

LR_PARAMS = sanitize_lgb_params(
    lr_run.get("best_params", {
        "objective": "lambdarank",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "num_leaves": 511,
        "learning_rate": 0.01,
        "min_child_samples": 200,
        "lambda_l1": 0.1,
        "lambda_l2": 1.0,
        "min_gain_to_split": 0.01,
        "feature_fraction": 0.8,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
    }),
    objective="lambdarank",
    metric="ndcg",
)
LR_PARAMS.update(gpu_params)
LR_PARAMS.update({"ndcg_eval_at": [5], "label_gain": [0, 1, 2, 3, 4, 5]})

XENDCG_PARAMS = dict(LR_PARAMS)
XENDCG_PARAMS.update({"objective": "rank_xendcg", "metric": "ndcg", "label_gain": [0, 1, 2, 3, 4, 5]})

UNBIASED_XENDCG_PARAMS = dict(XENDCG_PARAMS)
UNBIASED_XENDCG_PARAMS.update({"num_leaves": 255, "min_child_samples": 80, "feature_fraction": 0.75})

META_PARAMS = sanitize_lgb_params(
    {
        "objective": "rank_xendcg",
        "metric": "ndcg",
        "ndcg_eval_at": [5],
        "num_leaves": 15,
        "learning_rate": 0.03,
        "min_child_samples": 50,
        "lambda_l1": 0.0,
        "lambda_l2": 1.0,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
    },
    objective="rank_xendcg",
    metric="ndcg",
)
META_PARAMS.update(gpu_params)
META_PARAMS.update({"ndcg_eval_at": [5], "label_gain": [0, 1, 2, 3, 4, 5]})

print("Pointwise params:", PW_PARAMS)
print("Click params:", CLICK_PARAMS)
print("Click-only params:", CLICK_ONLY_PARAMS)
print("LambdaRank params:", LR_PARAMS)
print("RankXENDCG params:", XENDCG_PARAMS)
print("Unbiased RankXENDCG params:", UNBIASED_XENDCG_PARAMS)
print("Meta-ranker params:", META_PARAMS)


LightGBM device params: {'device': 'gpu', 'gpu_use_dp': False}
Pointwise params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 511, 'learning_rate': 0.01, 'min_child_samples': 200, 'lambda_l1': 0.1, 'lambda_l2': 1.0, 'min_gain_to_split': 0.01, 'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'feature_pre_filter': False, 'device': 'gpu', 'gpu_use_dp': False, 'verbose': -1, 'seed': 42, 'num_threads': -1}
Click params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 255, 'learning_rate': 0.03, 'min_child_samples': 150, 'lambda_l1': 0.05, 'lambda_l2': 1.0, 'feature_fraction': 0.7, 'bagging_fraction': 0.8, 'bagging_freq': 5, 'feature_pre_filter': False, 'verbose': -1, 'seed': 42, 'num_threads': -1, 'device': 'gpu', 'gpu_use_dp': False}
Click-only params: {'objective': 'binary', 'metric': 'binary_logloss', 'num_leaves': 255, 'learning_rate': 0.025, 'min_child_samples': 150, 'lambda_l1': 0.05, 'lambda_l2': 1.0, 'feature_fraction': 0.7,

## Helpers


In [4]:
META_ROUTING_CANDIDATES = [
    "search_hotel_count",
    "search_price_cv",
    "search_price_std",
    "search_brand_frac",
    "search_mean_star",
    "star_diff_from_search_mean",
    "search_promo_frac",
    "price_usd_rank",
    "price_rank_per_star",
    "price_z_in_tier",
    "price_vs_tier_median",
    "price_z_in_search",
    "prop_quality_score_rank",
    "srch_query_affinity_score_rank",
    "is_domestic",
    "is_family",
    "is_last_minute",
    "is_advance_purchase",
    "is_leisure_search",
    "last_minute_x_price_rank",
    "window_x_deal",
    "comp_advantage_score",
    "comp_advantage_rank",
    "comp_no_inventory_frac",
    "dest_star_book_rate_unbiased",
    "dest_star_ctr_unbiased",
    "dest_star_click_only_rate_unbiased",
    "hotel_ctr_by_window_unbiased",
    "hotel_book_rate_by_window_unbiased",
    "hotel_click_only_rate_by_window_unbiased",
    "svd_affinity_score",
    "hotel_book_rate_unbiased",
    "hotel_ctr_unbiased",
    "dest_hotel_affinity",
]


def available(cols, frame):
    return [c for c in cols if c in frame.columns]


def train_lgb(params, train_set, **kwargs):
    try:
        return lgb.train(params, train_set, **kwargs)
    except lgb.basic.LightGBMError as exc:
        if params.get("device") == "gpu" and LGB_AUTO_FALLBACK_TO_CPU:
            cpu_params = dict(params)
            cpu_params.pop("device", None)
            cpu_params.pop("gpu_use_dp", None)
            cpu_params["device_type"] = "cpu"
            print(f"LightGBM GPU failed; retrying on CPU: {exc}")
            return lgb.train(cpu_params, train_set, **kwargs)
        raise


def add_score_normalizations(frame, score_cols):
    frame = frame.copy()
    for col in score_cols:
        frame[f"{col}_z"] = z_normalize_per_search(frame, col)
        frame[f"{col}_mm"] = minmax_normalize_per_search(frame, col)
        frame[f"{col}_rank"] = frame.groupby("srch_id")[col].rank(
            ascending=False, pct=True, method="first"
        ).astype("float32")
    return frame


def train_binary_model(name, params, train_label, val_label, num_boost_round, early_stopping):
    train_set = lgb.Dataset(train_df[feature_cols], label=train_label)
    val_set = lgb.Dataset(val_df[feature_cols], label=val_label, reference=train_set)
    model = train_lgb(
        params,
        train_set,
        num_boost_round=num_boost_round,
        valid_sets=[val_set],
        callbacks=[lgb.early_stopping(early_stopping), lgb.log_evaluation(200)],
    )
    pred_df[name] = model.predict(val_df[feature_cols], num_iteration=model.best_iteration)
    ndcg = compute_ndcg_fast(pred_df, name)
    print(f"{name} NDCG@5: {ndcg:.6f}  trees={model.best_iteration}")
    return model, ndcg


def predict_sorted_model(model, sorted_frame, X, score_col):
    pred = sorted_frame[["srch_id", "prop_id"]].copy()
    pred[score_col] = model.predict(X, num_iteration=model.best_iteration)
    return pred


def prepare_meta_frame(frame, score_cols, routing_cols):
    meta = add_score_normalizations(frame.reset_index(drop=True), score_cols)
    norm_cols = []
    for col in score_cols:
        norm_cols.extend([f"{col}_z", f"{col}_mm", f"{col}_rank"])
    # Use per-search normalized score views to reduce final-time calibration shift.
    meta_feature_cols = norm_cols + routing_cols
    meta[meta_feature_cols] = meta[meta_feature_cols].replace([np.inf, -np.inf], np.nan)
    return meta, meta_feature_cols


def catboost_iteration_count(model, fallback):
    best_iter = model.get_best_iteration()
    if best_iter is None or int(best_iter) < 0:
        return int(fallback)
    return int(best_iter) + 1


def evaluate_oof_blend(frame, normalized_score_cols, signed=False, n_splits=5):
    groups = frame["srch_id"].to_numpy()
    n_groups = frame["srch_id"].nunique()
    splits = min(n_splits, n_groups)
    if splits < 2:
        return np.nan, None
    oof = np.zeros(len(frame), dtype="float32")
    fold_weights = []
    splitter = GroupKFold(n_splits=splits)
    for fit_idx, hold_idx in splitter.split(frame, groups=groups):
        fit_frame = frame.iloc[fit_idx]
        hold_frame = frame.iloc[hold_idx]
        if signed:
            fold_w, _ = optimize_signed_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        else:
            fold_w, _ = optimize_blend_weights(fit_frame, normalized_score_cols, maxiter=250)
        fold_weights.append(fold_w)
        blend = np.zeros(len(hold_frame), dtype="float64")
        for weight, col in zip(fold_w, normalized_score_cols):
            blend += weight * hold_frame[col].to_numpy(dtype="float64")
        oof[hold_idx] = blend.astype("float32")
    tmp = frame[["srch_id", "booking_bool", "click_bool"]].copy()
    tmp["_oof_blend"] = oof
    return compute_ndcg_fast(tmp, "_oof_blend"), fold_weights


def train_meta_oof(meta_df, meta_feature_cols):
    n_groups = meta_df["srch_id"].nunique()
    n_splits = min(META_N_SPLITS, n_groups)
    if n_splits < 2:
        raise ValueError("Need at least two search groups for meta OOF stacking")

    oof = np.zeros(len(meta_df), dtype="float32")
    best_iters = []
    fold_scores = []
    splitter = GroupKFold(n_splits=n_splits)

    for fold, (tr_idx, va_idx) in enumerate(splitter.split(meta_df, groups=meta_df["srch_id"]), start=1):
        fold_train = meta_df.iloc[tr_idx].copy()
        fold_val = meta_df.iloc[va_idx].copy()
        fold_val["_row_id"] = fold_val.index

        tr_sorted, X_tr, y_tr, group_tr = sort_for_group_model(
            fold_train, meta_feature_cols, make_relevance(fold_train)
        )
        va_sorted, X_va, y_va, group_va = sort_for_group_model(
            fold_val, meta_feature_cols, make_relevance(fold_val)
        )
        tr_set = lgb.Dataset(X_tr, label=y_tr, group=group_tr)
        va_set = lgb.Dataset(X_va, label=y_va, group=group_va, reference=tr_set)
        model = train_lgb(
            META_PARAMS,
            tr_set,
            num_boost_round=META_NUM_BOOST_ROUND,
            valid_sets=[va_set],
            callbacks=[lgb.early_stopping(META_EARLY_STOPPING), lgb.log_evaluation(0)],
        )
        preds = model.predict(X_va, num_iteration=model.best_iteration)
        oof[va_sorted["_row_id"].to_numpy()] = preds.astype("float32")
        best_iters.append(int(model.best_iteration or META_NUM_BOOST_ROUND))

        fold_eval = fold_val[["srch_id", "booking_bool", "click_bool"]].copy()
        fold_eval["_meta"] = oof[fold_val.index]
        fold_score = compute_ndcg_fast(fold_eval, "_meta")
        fold_scores.append(fold_score)
        print(f"Meta fold {fold}: NDCG@5={fold_score:.6f}, trees={best_iters[-1]}")

    meta_eval = meta_df[["srch_id", "booking_bool", "click_bool"]].copy()
    meta_eval["meta_oof_score"] = oof
    oof_score = compute_ndcg_fast(meta_eval, "meta_oof_score")
    full_rounds = int(np.median(best_iters)) if best_iters else 300

    all_sorted, X_all, y_all, group_all = sort_for_group_model(
        meta_df, meta_feature_cols, make_relevance(meta_df)
    )
    full_set = lgb.Dataset(X_all, label=y_all, group=group_all)
    full_model = train_lgb(META_PARAMS, full_set, num_boost_round=full_rounds)
    return oof, oof_score, full_model, full_rounds, fold_scores


## Build Validation Features


In [5]:
raw_df = load_train(sample_frac=SAMPLE_FRAC if USE_SAMPLE else None, random_state=RANDOM_STATE)
print(f"Raw rows: {len(raw_df):,}  searches: {raw_df['srch_id'].nunique():,}")

train_raw, val_raw = train_val_split(raw_df, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train rows: {len(train_raw):,}  Val rows: {len(val_raw):,}")
del raw_df
gc.collect()

count_reference_frame = train_raw
train_df, val_df = build_validation_feature_frames(
    train_raw,
    val_raw,
    count_reference_frame=count_reference_frame,
    svd_components=SVD_COMPONENTS,
)
del train_raw, val_raw, count_reference_frame
gc.collect()

assert train_df.duplicated(["srch_id", "prop_id"]).sum() == 0
assert val_df.duplicated(["srch_id", "prop_id"]).sum() == 0

feature_cols = get_feature_cols(train_df)
routing_cols = available(META_ROUTING_CANDIDATES, val_df)
print(f"Features: {len(feature_cols)}")
print(f"Meta routing features: {len(routing_cols)}")
print("Top NaN feature counts:")
print(train_df[feature_cols].isna().sum().sort_values(ascending=False).head(25))


Raw rows: 4,958,347  searches: 199,795
Train rows: 3,966,682  Val rows: 991,665
Features: 214
Meta routing features: 34
Top NaN feature counts:
visitor_hist_starrating             3764007
star_diff_visitor                   3764007
price_vs_visitor_hist               3763119
visitor_hist_adr_usd                3763119
visitor_hist_price_diff_rank        3763119
visitor_hist_price_diff             3763119
srch_query_affinity_score           3713645
srch_query_affinity_score_rank      3713645
comp_avg_diff                       2699463
comp_max_diff                       2699463
log_distance                        1282687
orig_destination_distance           1282687
distance_price_interaction          1282687
prop_location_score2_diff_mean       870478
prop_location_score2_diff_median     870478
prop_location_score2_rank            870478
prop_location_score2_std             149922
prop_location_score2_max              73926
prop_location_score2_min              73926
prop_review_score_ra

## Train Base Models


In [6]:
pred_df = val_df[["srch_id", "prop_id", "booking_bool", "click_bool"] + routing_cols].copy()

pw_model, pw_ndcg = train_binary_model(
    "pw_score",
    PW_PARAMS,
    train_df["booking_bool"].to_numpy(),
    val_df["booking_bool"].to_numpy(),
    PW_NUM_BOOST_ROUND,
    PW_EARLY_STOPPING,
)

click_model, click_ndcg = train_binary_model(
    "click_score",
    CLICK_PARAMS,
    train_df["click_bool"].to_numpy(),
    val_df["click_bool"].to_numpy(),
    CLICK_NUM_BOOST_ROUND,
    CLICK_EARLY_STOPPING,
)

train_click_only = (train_df["click_bool"] - train_df["booking_bool"]).clip(lower=0).to_numpy()
val_click_only = (val_df["click_bool"] - val_df["booking_bool"]).clip(lower=0).to_numpy()
click_only_model, click_only_ndcg = train_binary_model(
    "click_only_score",
    CLICK_ONLY_PARAMS,
    train_click_only,
    val_click_only,
    CLICK_ONLY_NUM_BOOST_ROUND,
    CLICK_ONLY_EARLY_STOPPING,
)


Training until validation scores don't improve for 150 rounds
[200]	valid_0's binary_logloss: 0.11118
[400]	valid_0's binary_logloss: 0.109344
[600]	valid_0's binary_logloss: 0.108551
[800]	valid_0's binary_logloss: 0.108098
[1000]	valid_0's binary_logloss: 0.107799
[1200]	valid_0's binary_logloss: 0.107613
[1400]	valid_0's binary_logloss: 0.107479
[1600]	valid_0's binary_logloss: 0.107373
[1800]	valid_0's binary_logloss: 0.107299
[2000]	valid_0's binary_logloss: 0.10724
[2200]	valid_0's binary_logloss: 0.107189
[2400]	valid_0's binary_logloss: 0.107152
[2600]	valid_0's binary_logloss: 0.107119
[2800]	valid_0's binary_logloss: 0.107102
[3000]	valid_0's binary_logloss: 0.10709
Early stopping, best iteration is:
[2998]	valid_0's binary_logloss: 0.107089
pw_score NDCG@5: 0.404184  trees=2998
Training until validation scores don't improve for 120 rounds
[200]	valid_0's binary_logloss: 0.16273
[400]	valid_0's binary_logloss: 0.161835
[600]	valid_0's binary_logloss: 0.161561
[800]	valid_0's 

In [7]:
train_sorted, X_train_rank, y_train_rank, train_group = sort_for_group_model(
    train_df, feature_cols, make_relevance(train_df)
)
val_sorted, X_val_rank, y_val_rank, val_group = sort_for_group_model(
    val_df, feature_cols, make_relevance(val_df)
)

rank_train = lgb.Dataset(X_train_rank, label=y_train_rank, group=train_group)
rank_val = lgb.Dataset(X_val_rank, label=y_val_rank, group=val_group, reference=rank_train)

lr_model = train_lgb(
    LR_PARAMS,
    rank_train,
    num_boost_round=LR_NUM_BOOST_ROUND,
    valid_sets=[rank_val],
    callbacks=[lgb.early_stopping(LR_EARLY_STOPPING), lgb.log_evaluation(200)],
)
lr_pred = predict_sorted_model(lr_model, val_sorted, X_val_rank, "lr_score")
pred_df = pred_df.merge(lr_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
lr_ndcg = compute_ndcg_fast(pred_df, "lr_score")
print(f"LambdaRank NDCG@5: {lr_ndcg:.6f}  trees={lr_model.best_iteration}")


Training until validation scores don't improve for 250 rounds
[200]	valid_0's ndcg@5: 0.377457
[400]	valid_0's ndcg@5: 0.385695
[600]	valid_0's ndcg@5: 0.390961
[800]	valid_0's ndcg@5: 0.39401
[1000]	valid_0's ndcg@5: 0.394972
[1200]	valid_0's ndcg@5: 0.396096
[1400]	valid_0's ndcg@5: 0.396634
[1600]	valid_0's ndcg@5: 0.397467
[1800]	valid_0's ndcg@5: 0.397179
Early stopping, best iteration is:
[1679]	valid_0's ndcg@5: 0.397706
LambdaRank NDCG@5: 0.397706  trees=1679


In [8]:
xendcg_model = None
xendcg_ndcg = None
if RUN_RANK_XENDCG:
    xendcg_model = train_lgb(
        XENDCG_PARAMS,
        rank_train,
        num_boost_round=XENDCG_NUM_BOOST_ROUND,
        valid_sets=[rank_val],
        callbacks=[lgb.early_stopping(XENDCG_EARLY_STOPPING), lgb.log_evaluation(200)],
    )
    xendcg_pred = predict_sorted_model(xendcg_model, val_sorted, X_val_rank, "xendcg_score")
    pred_df = pred_df.merge(xendcg_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
    xendcg_ndcg = compute_ndcg_fast(pred_df, "xendcg_score")
    print(f"RankXENDCG NDCG@5: {xendcg_ndcg:.6f}  trees={xendcg_model.best_iteration}")


Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.381862
[400]	valid_0's ndcg@5: 0.390699
[600]	valid_0's ndcg@5: 0.395766
[800]	valid_0's ndcg@5: 0.399269
[1000]	valid_0's ndcg@5: 0.401712
[1200]	valid_0's ndcg@5: 0.403214
[1400]	valid_0's ndcg@5: 0.404205
[1600]	valid_0's ndcg@5: 0.404659
[1800]	valid_0's ndcg@5: 0.405407
Early stopping, best iteration is:
[1793]	valid_0's ndcg@5: 0.405659
RankXENDCG NDCG@5: 0.405659  trees=1793


In [9]:
ub_xendcg_model = None
ub_xendcg_ndcg = None
if RUN_UNBIASED_XENDCG:
    ub_train_df = train_df[train_df["random_bool"] == 1].copy()
    ub_sorted, X_ub_train, y_ub_train, ub_group = sort_for_group_model(
        ub_train_df, feature_cols, make_relevance(ub_train_df)
    )
    ub_train_set = lgb.Dataset(X_ub_train, label=y_ub_train, group=ub_group)
    ub_val_set = lgb.Dataset(X_val_rank, label=y_val_rank, group=val_group, reference=ub_train_set)
    ub_xendcg_model = train_lgb(
        UNBIASED_XENDCG_PARAMS,
        ub_train_set,
        num_boost_round=UNBIASED_XENDCG_NUM_BOOST_ROUND,
        valid_sets=[ub_val_set],
        callbacks=[lgb.early_stopping(UNBIASED_XENDCG_EARLY_STOPPING), lgb.log_evaluation(200)],
    )
    ub_pred = predict_sorted_model(ub_xendcg_model, val_sorted, X_val_rank, "ub_xendcg_score")
    pred_df = pred_df.merge(ub_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
    ub_xendcg_ndcg = compute_ndcg_fast(pred_df, "ub_xendcg_score")
    print(f"Unbiased RankXENDCG NDCG@5: {ub_xendcg_ndcg:.6f}  trees={ub_xendcg_model.best_iteration}")
    del ub_train_df, ub_sorted, X_ub_train, y_ub_train, ub_group, ub_train_set, ub_val_set
    gc.collect()


Training until validation scores don't improve for 200 rounds
[200]	valid_0's ndcg@5: 0.250178
[400]	valid_0's ndcg@5: 0.261185
[600]	valid_0's ndcg@5: 0.268433
[800]	valid_0's ndcg@5: 0.272467
[1000]	valid_0's ndcg@5: 0.274412
[1200]	valid_0's ndcg@5: 0.276267
[1400]	valid_0's ndcg@5: 0.276737
[1600]	valid_0's ndcg@5: 0.276962
[1800]	valid_0's ndcg@5: 0.277824
[2000]	valid_0's ndcg@5: 0.277565
Early stopping, best iteration is:
[1925]	valid_0's ndcg@5: 0.277977
Unbiased RankXENDCG NDCG@5: 0.277977  trees=1925


In [10]:
cat_model = None
cat_ndcg = None
if RUN_CATBOOST and CatBoostRanker is not None:
    cb_feature_cols = get_feature_cols(train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
    cb_train, X_cb_train, y_cb_train, cb_train_group = sort_for_group_model(
        train_df, cb_feature_cols, make_relevance(train_df)
    )
    cb_val, X_cb_val, y_cb_val, cb_val_group = sort_for_group_model(
        val_df, cb_feature_cols, make_relevance(val_df)
    )

    cat_features_names = []
    if CATBOOST_USE_ID_CATEGORICALS:
        cat_features_names.extend(["prop_id", "srch_destination_id"])
    cat_features_names.extend(["site_id", "visitor_location_country_id", "prop_country_id"])
    cat_features_names = [c for c in dict.fromkeys(cat_features_names) if c in cb_feature_cols]
    cat_features_idx = [cb_feature_cols.index(c) for c in cat_features_names]
    assert len(cat_features_idx) == len(cat_features_names), (
        f"Missing CatBoost categoricals: {set(cat_features_names) - set(cb_feature_cols)}"
    )

    cb_train_pool = Pool(
        X_cb_train,
        label=y_cb_train,
        group_id=cb_train["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    cb_val_pool = Pool(
        X_cb_val,
        label=y_cb_val,
        group_id=cb_val["srch_id"].to_numpy(),
        cat_features=cat_features_idx,
        feature_names=cb_feature_cols,
    )
    cat_model = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=5;type=Base",
        iterations=CATBOOST_ITERATIONS,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE,
        task_type=CATBOOST_TASK_TYPE,
        verbose=200,
        allow_writing_files=False,
    )
    cat_model.fit(
        cb_train_pool,
        eval_set=cb_val_pool,
        early_stopping_rounds=CATBOOST_EARLY_STOPPING,
        use_best_model=True,
    )
    cb_pred = cb_val[["srch_id", "prop_id"]].copy()
    cb_pred["cb_score"] = cat_model.predict(cb_val_pool)
    pred_df = pred_df.merge(cb_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
    cat_ndcg = compute_ndcg_fast(pred_df, "cb_score")
    print(f"CatBoost NDCG@5: {cat_ndcg:.6f}  trees={cat_model.get_best_iteration()}")
    del cb_train_pool, cb_val_pool, cb_train, cb_val, X_cb_train, X_cb_val, y_cb_train, y_cb_val
    gc.collect()
else:
    print("Skipping CatBoost")


Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2874105	best: 0.2874105 (0)	total: 1.02s	remaining: 1h 41m 37s
200:	test: 0.3558247	best: 0.3558247 (200)	total: 3m 21s	remaining: 1h 36m 46s
400:	test: 0.3673591	best: 0.3673591 (400)	total: 6m 31s	remaining: 1h 31m 9s
600:	test: 0.3725564	best: 0.3725690 (599)	total: 9m 37s	remaining: 1h 26m 26s
800:	test: 0.3759442	best: 0.3759468 (789)	total: 12m 37s	remaining: 1h 21m 59s
1000:	test: 0.3788393	best: 0.3788588 (997)	total: 15m 36s	remaining: 1h 17m 58s
1200:	test: 0.3809151	best: 0.3809151 (1200)	total: 18m 33s	remaining: 1h 14m 11s
1400:	test: 0.3827567	best: 0.3827897 (1399)	total: 21m 29s	remaining: 1h 10m 33s
1600:	test: 0.3843566	best: 0.3844413 (1586)	total: 24m 25s	remaining: 1h 7m 6s
1800:	test: 0.3856368	best: 0.3856617 (1795)	total: 27m 21s	remaining: 1h 3m 47s
2000:	test: 0.3866284	best: 0.3866284 (2000)	total: 30m 16s	remaining: 1h 30s
2200:	test: 0.3875600	best: 0.3875877 (2198)	total: 33m 12s	remaining: 57m 19s
2400:	test: 0.3889152	best: 0.3889438 (2399)	to

## Blend And Stack

The ordinary blends are kept as baselines. The signed blend can subtract click-only if validation supports it. The meta-ranker is the main new combiner: it uses OOF predictions across validation searches, so its reported score is not trained on the same searches it scores.


In [11]:
score_cols = [
    c for c in [
        "pw_score",
        "click_score",
        "click_only_score",
        "lr_score",
        "xendcg_score",
        "ub_xendcg_score",
        "cb_score",
    ]
    if c in pred_df.columns
]
base_scores = {col: compute_ndcg_fast(pred_df, col) for col in score_cols}
print("Base scores:")
for col, score in sorted(base_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")

pred_df = add_score_normalizations(pred_df, score_cols)
pred_df["rrf_score"] = rrf_score(pred_df, score_cols, k=60)
rrf_ndcg = compute_ndcg_fast(pred_df, "rrf_score")
print(f"RRF NDCG@5: {rrf_ndcg:.6f}")

z_cols = [f"{col}_z" for col in score_cols]
weights, opt_ndcg = optimize_blend_weights(pred_df, z_cols)
pred_df["opt_z_blend"] = sum(w * pred_df[col] for w, col in zip(weights, z_cols))
print("Optimized nonnegative z-blend in-sample:", opt_ndcg, dict(zip(score_cols, weights.round(4))))
opt_z_oof_ndcg, opt_z_oof_weights = evaluate_oof_blend(pred_df, z_cols, signed=False, n_splits=META_N_SPLITS)
print(f"Optimized nonnegative z-blend OOF: {opt_z_oof_ndcg:.6f}")

mm_cols = [f"{col}_mm" for col in score_cols]
mm_weights, mm_ndcg = optimize_blend_weights(pred_df, mm_cols)
pred_df["opt_mm_blend"] = sum(w * pred_df[col] for w, col in zip(mm_weights, mm_cols))
print("Optimized nonnegative minmax blend in-sample:", mm_ndcg, dict(zip(score_cols, mm_weights.round(4))))
opt_mm_oof_ndcg, opt_mm_oof_weights = evaluate_oof_blend(pred_df, mm_cols, signed=False, n_splits=META_N_SPLITS)
print(f"Optimized nonnegative minmax blend OOF: {opt_mm_oof_ndcg:.6f}")

signed_weights, signed_ndcg = optimize_signed_blend_weights(pred_df, z_cols)
pred_df["signed_z_blend"] = sum(w * pred_df[col] for w, col in zip(signed_weights, z_cols))
print("Optimized signed z-blend in-sample:", signed_ndcg, dict(zip(score_cols, signed_weights.round(4))))
signed_oof_ndcg, signed_oof_weights = evaluate_oof_blend(pred_df, z_cols, signed=True, n_splits=META_N_SPLITS)
print(f"Optimized signed z-blend OOF: {signed_oof_ndcg:.6f}")


Base scores:
  xendcg_score: 0.405659
  pw_score: 0.404184
  lr_score: 0.397706
  cb_score: 0.397648
  click_score: 0.393025
  ub_xendcg_score: 0.277977
  click_only_score: 0.276174
RRF NDCG@5: 0.386912
Optimized nonnegative z-blend in-sample: 0.40775673076988567 {'pw_score': np.float64(0.2256), 'click_score': np.float64(0.1107), 'click_only_score': np.float64(0.001), 'lr_score': np.float64(0.1856), 'xendcg_score': np.float64(0.2634), 'ub_xendcg_score': np.float64(0.0), 'cb_score': np.float64(0.2137)}
Optimized nonnegative z-blend OOF: 0.406709
Optimized nonnegative minmax blend in-sample: 0.4069477085295072 {'pw_score': np.float64(0.2924), 'click_score': np.float64(0.1569), 'click_only_score': np.float64(0.0016), 'lr_score': np.float64(0.0997), 'xendcg_score': np.float64(0.2009), 'ub_xendcg_score': np.float64(0.0001), 'cb_score': np.float64(0.2485)}
Optimized nonnegative minmax blend OOF: 0.405304
Optimized signed z-blend in-sample: 0.4099110828950275 {'pw_score': np.float64(0.1561), 

In [12]:
meta_model_full = None
meta_feature_cols = []
meta_oof_ndcg = None
meta_full_rounds = None
meta_fold_scores = []
if RUN_META_STACKER:
    meta_df, meta_feature_cols = prepare_meta_frame(pred_df, score_cols, routing_cols)
    meta_oof, meta_oof_ndcg, meta_model_full, meta_full_rounds, meta_fold_scores = train_meta_oof(
        meta_df,
        meta_feature_cols,
    )
    pred_df["meta_oof_score"] = meta_oof
    print(f"Meta OOF NDCG@5: {meta_oof_ndcg:.6f}  final_rounds={meta_full_rounds}")
    print("Meta features:", meta_feature_cols)
else:
    print("Skipping meta stacker")


Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[164]	valid_0's ndcg@5: 0.411569
Meta fold 1: NDCG@5=0.411569, trees=164
Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[376]	valid_0's ndcg@5: 0.409912
Meta fold 2: NDCG@5=0.409912, trees=376
Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[268]	valid_0's ndcg@5: 0.415226
Meta fold 3: NDCG@5=0.415226, trees=268
Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[171]	valid_0's ndcg@5: 0.401237
Meta fold 4: NDCG@5=0.401237, trees=171
Training until validation scores don't improve for 75 rounds
Early stopping, best iteration is:
[43]	valid_0's ndcg@5: 0.412326
Meta fold 5: NDCG@5=0.412326, trees=43
Meta OOF NDCG@5: 0.410054  final_rounds=171
Meta features: ['pw_score_z', 'pw_score_mm', 'pw_score_rank', 'click_score_z', 'click_score_mm', 'click_score_ran

In [ ]:
ensemble_scores_in_sample = {
    "rrf_score": compute_ndcg_fast(pred_df, "rrf_score"),
    "opt_z_blend": compute_ndcg_fast(pred_df, "opt_z_blend"),
    "opt_mm_blend": compute_ndcg_fast(pred_df, "opt_mm_blend"),
    "signed_z_blend": compute_ndcg_fast(pred_df, "signed_z_blend"),
}
ensemble_scores = {
    "rrf_score": ensemble_scores_in_sample["rrf_score"],
    "opt_z_blend": float(opt_z_oof_ndcg),
    "opt_mm_blend": float(opt_mm_oof_ndcg),
    "signed_z_blend": float(signed_oof_ndcg),
}
if "meta_oof_score" in pred_df.columns:
    ensemble_scores["meta_oof_score"] = float(meta_oof_ndcg)
best_ensemble_col = max(ensemble_scores, key=ensemble_scores.get)
best_ensemble_ndcg = ensemble_scores[best_ensemble_col]
print("Ensemble scores used for selection:")
for col, score in sorted(ensemble_scores.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")
print("Ensemble in-sample diagnostics:")
for col, score in sorted(ensemble_scores_in_sample.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {col}: {score:.6f}")
print(f"Best ensemble: {best_ensemble_col}  selection_score={best_ensemble_ndcg:.6f}")

run_payload = {
    "use_sample": USE_SAMPLE,
    "sample_frac": SAMPLE_FRAC if USE_SAMPLE else None,
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "n_features": int(len(feature_cols)),
    "n_meta_routing_features": int(len(routing_cols)),
    "score_cols": score_cols,
    "base_scores": {k: float(v) for k, v in base_scores.items()},
    "rrf_ndcg5": float(rrf_ndcg),
    "opt_z_ndcg5_in_sample": float(opt_ndcg),
    "opt_z_ndcg5_oof": float(opt_z_oof_ndcg),
    "opt_z_weights": dict(zip(score_cols, [float(w) for w in weights])),
    "opt_mm_ndcg5_in_sample": float(mm_ndcg),
    "opt_mm_ndcg5_oof": float(opt_mm_oof_ndcg),
    "opt_mm_weights": dict(zip(score_cols, [float(w) for w in mm_weights])),
    "signed_z_ndcg5_in_sample": float(signed_ndcg),
    "signed_z_ndcg5_oof": float(signed_oof_ndcg),
    "signed_z_weights": dict(zip(score_cols, [float(w) for w in signed_weights])),
    "ensemble_scores_in_sample": {k: float(v) for k, v in ensemble_scores_in_sample.items()},
    "ensemble_scores_selection": {k: float(v) for k, v in ensemble_scores.items()},
    "meta_oof_ndcg5": None if meta_oof_ndcg is None else float(meta_oof_ndcg),
    "meta_full_rounds": None if meta_full_rounds is None else int(meta_full_rounds),
    "meta_fold_scores": [float(x) for x in meta_fold_scores],
    "meta_feature_cols": meta_feature_cols,
    "best_ensemble_col": best_ensemble_col,
    "best_ensemble_ndcg5": float(best_ensemble_ndcg),
    "best_iterations": {
        "pointwise": int(pw_model.best_iteration or PW_NUM_BOOST_ROUND),
        "click": int(click_model.best_iteration or CLICK_NUM_BOOST_ROUND),
        "click_only": int(click_only_model.best_iteration or CLICK_ONLY_NUM_BOOST_ROUND),
        "lambdarank": int(lr_model.best_iteration or LR_NUM_BOOST_ROUND),
        "rank_xendcg": None if xendcg_model is None else int(xendcg_model.best_iteration or XENDCG_NUM_BOOST_ROUND),
        "unbiased_rank_xendcg": None if ub_xendcg_model is None else int(ub_xendcg_model.best_iteration or UNBIASED_XENDCG_NUM_BOOST_ROUND),
        "catboost": None if cat_model is None else catboost_iteration_count(cat_model, CATBOOST_ITERATIONS),
    },
}
save_run_json(OUTPUT_DIR / "afterburner_validation_run.json", run_payload)
pred_df.to_parquet(OUTPUT_DIR / "afterburner_validation_predictions.parquet", index=False)
run_payload


Ensemble scores used for selection:
  meta_oof_score: 0.410054
  signed_z_blend: 0.408544
  opt_z_blend: 0.406709
  opt_mm_blend: 0.405304
  rrf_score: 0.386912
Ensemble in-sample diagnostics:
  signed_z_blend: 0.409911
  opt_z_blend: 0.407757
  opt_mm_blend: 0.406948
  rrf_score: 0.386912
Best ensemble: meta_oof_score  selection_score=0.410054


{'use_sample': False,
 'sample_frac': None,
 'n_train': 3966682,
 'n_val': 991665,
 'n_features': 214,
 'n_meta_routing_features': 34,
 'score_cols': ['pw_score',
  'click_score',
  'click_only_score',
  'lr_score',
  'xendcg_score',
  'ub_xendcg_score',
  'cb_score'],
 'base_scores': {'pw_score': 0.404183567850474,
  'click_score': 0.39302461633473323,
  'click_only_score': 0.27617417874926675,
  'lr_score': 0.3977057669240045,
  'xendcg_score': 0.40565895465911744,
  'ub_xendcg_score': 0.2779774553657641,
  'cb_score': 0.3976480959654903},
 'rrf_ndcg5': 0.38691166543583033,
 'opt_z_ndcg5_in_sample': 0.40775673076988567,
 'opt_z_ndcg5_oof': 0.4067090157522559,
 'opt_z_weights': {'pw_score': 0.22558514768355553,
  'click_score': 0.11070336298773745,
  'click_only_score': 0.0009822234324190356,
  'lr_score': 0.1855813484163933,
  'xendcg_score': 0.26342439773047194,
  'ub_xendcg_score': 2.7681873278763062e-05,
  'cb_score': 0.21369583787614405},
 'opt_mm_ndcg5_in_sample': 0.406947708529

## Optional Final Submission Retrain

Set `RUN_FINAL_SUBMISSION=True` only after the validation run has selected the final ensemble. The final cell can use in-memory validation artifacts, or reload `outputs/afterburner_validation_run.json` plus saved validation predictions after a kernel restart.


In [ ]:
if not RUN_FINAL_SUBMISSION:
    print("RUN_FINAL_SUBMISSION=False; validation-only run complete.")
else:
    validation_payload = globals().get("run_payload")
    if validation_payload is None:
        with (OUTPUT_DIR / "afterburner_validation_run.json").open() as f:
            validation_payload = json.load(f)

    score_cols = list(globals().get("score_cols", validation_payload["score_cols"]))
    best_iterations = dict(validation_payload.get("best_iterations", {}))
    best_ensemble_col = globals().get("best_ensemble_col", validation_payload["best_ensemble_col"])
    weights = globals().get(
        "weights",
        np.array([validation_payload["opt_z_weights"][c] for c in score_cols], dtype="float64"),
    )
    mm_weights = globals().get(
        "mm_weights",
        np.array([validation_payload["opt_mm_weights"][c] for c in score_cols], dtype="float64"),
    )
    signed_weights = globals().get(
        "signed_weights",
        np.array([validation_payload["signed_z_weights"][c] for c in score_cols], dtype="float64"),
    )
    meta_feature_cols = globals().get("meta_feature_cols", validation_payload.get("meta_feature_cols", []))
    meta_model_full = globals().get("meta_model_full")

    if best_ensemble_col == "meta_oof_score" and meta_model_full is None:
        saved_pred_path = OUTPUT_DIR / "afterburner_validation_predictions.parquet"
        saved_pred_df = pd.read_parquet(saved_pred_path)
        missing_meta = set(meta_feature_cols) - set(saved_pred_df.columns)
        assert not missing_meta, f"Saved validation predictions missing meta features: {missing_meta}"
        saved_pred_df[meta_feature_cols] = saved_pred_df[meta_feature_cols].replace([np.inf, -np.inf], np.nan)
        meta_sorted, X_meta_full, y_meta_full, meta_group_full = sort_for_group_model(
            saved_pred_df,
            meta_feature_cols,
            make_relevance(saved_pred_df),
        )
        meta_model_full = train_lgb(
            META_PARAMS,
            lgb.Dataset(X_meta_full, label=y_meta_full, group=meta_group_full),
            num_boost_round=int(validation_payload.get("meta_full_rounds") or 300),
        )
        del saved_pred_df, meta_sorted, X_meta_full, y_meta_full, meta_group_full
        gc.collect()

    for _name in [
        "train_df", "val_df", "train_sorted", "val_sorted", "X_train_rank", "X_val_rank",
        "rank_train", "rank_val", "meta_df", "pred_df",
    ]:
        if _name in globals():
            del globals()[_name]
    gc.collect()

    full_train_raw = load_train(sample_frac=None, random_state=RANDOM_STATE)
    test_raw = load_test()

    if USE_TRAIN_TEST_COUNTS:
        count_reference_frame = pd.concat(
            [
                full_train_raw[["prop_id", "srch_destination_id"]],
                test_raw[["prop_id", "srch_destination_id"]],
            ],
            ignore_index=True,
        )
    else:
        count_reference_frame = full_train_raw

    final_train_df, test_df = build_test_feature_frame(
        full_train_raw,
        test_raw,
        count_reference_frame=count_reference_frame,
        svd_components=SVD_COMPONENTS,
    )
    del full_train_raw, test_raw, count_reference_frame
    gc.collect()

    assert test_df.duplicated(["srch_id", "prop_id"]).sum() == 0
    final_feature_cols = get_feature_cols(final_train_df)
    final_routing_cols = available(META_ROUTING_CANDIDATES, test_df)
    final_scores = test_df[["srch_id", "prop_id"] + final_routing_cols].copy()

    final_pw = train_lgb(
        PW_PARAMS,
        lgb.Dataset(final_train_df[final_feature_cols], label=final_train_df["booking_bool"].to_numpy()),
        num_boost_round=int(best_iterations.get("pointwise") or PW_NUM_BOOST_ROUND),
    )
    final_scores["pw_score"] = final_pw.predict(test_df[final_feature_cols])

    final_click = train_lgb(
        CLICK_PARAMS,
        lgb.Dataset(final_train_df[final_feature_cols], label=final_train_df["click_bool"].to_numpy()),
        num_boost_round=int(best_iterations.get("click") or CLICK_NUM_BOOST_ROUND),
    )
    final_scores["click_score"] = final_click.predict(test_df[final_feature_cols])

    final_click_only_label = (final_train_df["click_bool"] - final_train_df["booking_bool"]).clip(lower=0).to_numpy()
    final_click_only = train_lgb(
        CLICK_ONLY_PARAMS,
        lgb.Dataset(final_train_df[final_feature_cols], label=final_click_only_label),
        num_boost_round=int(best_iterations.get("click_only") or CLICK_ONLY_NUM_BOOST_ROUND),
    )
    final_scores["click_only_score"] = final_click_only.predict(test_df[final_feature_cols])

    full_sorted, X_full_rank, y_full_rank, full_group = sort_for_group_model(
        final_train_df, final_feature_cols, make_relevance(final_train_df)
    )
    test_sorted, X_test_rank, _, _ = sort_for_group_model(test_df, final_feature_cols)
    full_rank_set = lgb.Dataset(X_full_rank, label=y_full_rank, group=full_group)

    final_lr = train_lgb(
        LR_PARAMS,
        full_rank_set,
        num_boost_round=int(best_iterations.get("lambdarank") or LR_NUM_BOOST_ROUND),
    )
    lr_test_pred = test_sorted[["srch_id", "prop_id"]].copy()
    lr_test_pred["lr_score"] = final_lr.predict(X_test_rank)
    final_scores = final_scores.merge(lr_test_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

    if "xendcg_score" in score_cols:
        final_xendcg = train_lgb(
            XENDCG_PARAMS,
            full_rank_set,
            num_boost_round=int(best_iterations.get("rank_xendcg") or XENDCG_NUM_BOOST_ROUND),
        )
        xendcg_test_pred = test_sorted[["srch_id", "prop_id"]].copy()
        xendcg_test_pred["xendcg_score"] = final_xendcg.predict(X_test_rank)
        final_scores = final_scores.merge(xendcg_test_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")

    if "ub_xendcg_score" in score_cols:
        ub_final_train_df = final_train_df[final_train_df["random_bool"] == 1].copy()
        ub_full_sorted, X_ub_full, y_ub_full, ub_full_group = sort_for_group_model(
            ub_final_train_df, final_feature_cols, make_relevance(ub_final_train_df)
        )
        ub_full_set = lgb.Dataset(X_ub_full, label=y_ub_full, group=ub_full_group)
        final_ub = train_lgb(
            UNBIASED_XENDCG_PARAMS,
            ub_full_set,
            num_boost_round=int(best_iterations.get("unbiased_rank_xendcg") or UNBIASED_XENDCG_NUM_BOOST_ROUND),
        )
        ub_test_pred = test_sorted[["srch_id", "prop_id"]].copy()
        ub_test_pred["ub_xendcg_score"] = final_ub.predict(X_test_rank)
        final_scores = final_scores.merge(ub_test_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
        del ub_final_train_df, ub_full_sorted, X_ub_full, y_ub_full, ub_full_group, ub_full_set
        gc.collect()

    if "cb_score" in score_cols:
        cb_feature_cols = get_feature_cols(final_train_df, keep_svd_factors=CATBOOST_KEEP_SVD_FACTORS)
        cb_full, X_cb_full, y_cb_full, _ = sort_for_group_model(
            final_train_df, cb_feature_cols, make_relevance(final_train_df)
        )
        cb_test, X_cb_test, _, _ = sort_for_group_model(test_df, cb_feature_cols)
        cat_features_names = []
        if CATBOOST_USE_ID_CATEGORICALS:
            cat_features_names.extend(["prop_id", "srch_destination_id"])
        cat_features_names.extend(["site_id", "visitor_location_country_id", "prop_country_id"])
        cat_features_names = [c for c in dict.fromkeys(cat_features_names) if c in cb_feature_cols]
        cat_features_idx = [cb_feature_cols.index(c) for c in cat_features_names]
        assert len(cat_features_idx) == len(cat_features_names), (
            f"Missing CatBoost categoricals: {set(cat_features_names) - set(cb_feature_cols)}"
        )
        cb_full_pool = Pool(
            X_cb_full,
            label=y_cb_full,
            group_id=cb_full["srch_id"].to_numpy(),
            cat_features=cat_features_idx,
            feature_names=cb_feature_cols,
        )
        cb_test_pool = Pool(
            X_cb_test,
            group_id=cb_test["srch_id"].to_numpy(),
            cat_features=cat_features_idx,
            feature_names=cb_feature_cols,
        )
        assert CatBoostRanker is not None and Pool is not None, "CatBoost is required because cb_score won/was selected"
        catboost_iters = int(best_iterations.get("catboost") or CATBOOST_ITERATIONS)
        final_cb_params = {
            "loss_function": "YetiRank",
            "eval_metric": "NDCG:top=5;type=Base",
            "iterations": catboost_iters,
            "learning_rate": 0.03,
            "depth": 8,
            "l2_leaf_reg": 3.0,
            "random_seed": RANDOM_STATE,
            "task_type": CATBOOST_TASK_TYPE,
            "verbose": 200,
            "allow_writing_files": False,
        }
        if "cat_model" in globals() and cat_model is not None:
            fit_only_keys = {"early_stopping_rounds", "use_best_model", "eval_set"}
            model_params = {
                k: v for k, v in cat_model.get_params().items()
                if k not in fit_only_keys
            }
            final_cb_params.update(model_params)
        final_cb_params["loss_function"] = "YetiRank"
        final_cb_params["iterations"] = catboost_iters
        assert final_cb_params.get("loss_function") == "YetiRank", final_cb_params
        final_cb = CatBoostRanker(**final_cb_params)
        final_cb.fit(cb_full_pool)
        cb_test_pred = cb_test[["srch_id", "prop_id"]].copy()
        cb_test_pred["cb_score"] = final_cb.predict(cb_test_pool)
        final_scores = final_scores.merge(cb_test_pred, on=["srch_id", "prop_id"], how="left", validate="one_to_one")
        del cb_full_pool, cb_test_pool, cb_full, cb_test, X_cb_full, X_cb_test, y_cb_full
        gc.collect()

    missing_scores = [c for c in score_cols if c not in final_scores.columns]
    assert not missing_scores, f"Final scores missing validation-selected models: {missing_scores}"
    final_score_cols = list(score_cols)
    final_scores = add_score_normalizations(final_scores, final_score_cols)

    if best_ensemble_col == "rrf_score":
        final_scores["final_score"] = rrf_score(final_scores, final_score_cols, k=60)
    elif best_ensemble_col == "opt_z_blend":
        weight_by_score = dict(zip(score_cols, weights))
        final_scores["final_score"] = sum(
            weight_by_score[col] * final_scores[f"{col}_z"] for col in final_score_cols
        )
    elif best_ensemble_col == "opt_mm_blend":
        weight_by_score = dict(zip(score_cols, mm_weights))
        final_scores["final_score"] = sum(
            weight_by_score[col] * final_scores[f"{col}_mm"] for col in final_score_cols
        )
    elif best_ensemble_col == "signed_z_blend":
        weight_by_score = dict(zip(score_cols, signed_weights))
        final_scores["final_score"] = sum(
            weight_by_score[col] * final_scores[f"{col}_z"] for col in final_score_cols
        )
    elif best_ensemble_col == "meta_oof_score":
        assert meta_model_full is not None
        final_meta_features = meta_feature_cols
        missing = set(final_meta_features) - set(final_scores.columns)
        assert not missing, f"Missing final meta features: {missing}"
        final_scores[final_meta_features] = final_scores[final_meta_features].replace([np.inf, -np.inf], np.nan)
        final_scores["final_score"] = meta_model_full.predict(final_scores[final_meta_features])
    else:
        raise ValueError(f"Unknown best ensemble: {best_ensemble_col}")

    sub_path = ROOT / "submission_afterburner.csv"
    sub = make_submission(final_scores, "final_score", sub_path)
    assert len(sub) == len(final_scores)
    assert sub["srch_id"].is_monotonic_increasing
    print(f"Wrote {sub_path} rows={len(sub):,}")


Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 1.27s	remaining: 2h 6m 58s
200:	total: 4m 12s	remaining: 2h 57s
400:	total: 8m 9s	remaining: 1h 53m 33s
600:	total: 11m 59s	remaining: 1h 47m 29s
800:	total: 15m 45s	remaining: 1h 42m
1000:	total: 19m 28s	remaining: 1h 36m 59s
1200:	total: 23m 9s	remaining: 1h 32m 17s
1400:	total: 26m 48s	remaining: 1h 27m 47s
1600:	total: 30m 27s	remaining: 1h 23m 26s
1800:	total: 34m 6s	remaining: 1h 19m 17s
2000:	total: 37m 45s	remaining: 1h 15m 14s
2200:	total: 41m 24s	remaining: 1h 11m 14s
2400:	total: 45m 2s	remaining: 1h 7m 17s
2600:	total: 48m 43s	remaining: 1h 3m 27s
2800:	total: 52m 23s	remaining: 59m 36s
3000:	total: 56m 2s	remaining: 55m 46s
3200:	total: 59m 41s	remaining: 51m 58s
3400:	total: 1h 3m 21s	remaining: 48m 11s
3600:	total: 1h 7m 1s	remaining: 44m 25s
3800:	total: 1h 10m 40s	remaining: 40m 39s
4000:	total: 1h 14m 21s	remaining: 36m 55s
4200:	total: 1h 18m 1s	remaining: 33m 11s
4400:	total: 1h 21m 41s	remaining: 29m 27s
4600:	total: 1h 25m 21s	remaining: 25m 43s
4800:	to